# Importing the Libraries


In [ ]:
import requests
import pandas as pd
import numpy as np

# Getting the data using API

In [ ]:


# API endpoint
url = "https://ssd-api.jpl.nasa.gov/cad.api"

# Optional parameters (adjust as needed)
params = {
    "date-min": "1900-01-01",  # Start date
    "date-max": "2025-12-31",   # End date
    "dist-max": "0.2",          # Max distance (Lunar Distance)
}

# Send GET request
response = requests.get(url, params=params)
data = response.json()  # Parse JSON response

# Convert to DataFrame
df = pd.DataFrame(data["data"], columns=data["fields"])
print(df.head())

# Inspecting the data

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head(4)

In [ ]:
df.info()

In [ ]:
missing_h = df['h'].isna().sum()
missing_h

In [ ]:
null_h_indices=df[df['h'].isna()].index
null_h_indices

# Renaming the headers

In [ ]:
df=df.rename(columns={'des':'Object ID', 'orbit_id':'Orbit ID', 'jd':'Julian Date', 'cd':'Close Approach Date', 'dist':'Nominal Distance', 'dist_min':'Min Distance', 'dist_max':'Max Distance', 'v_rel':'Relative Velocity',
       'v_inf':'Infinity Velocity', 't_sigma_f':'Time Uncertainity', 'h':'Magnitude H'})
df

# Column Wise Inspecting

**1.Object and Orbit**

In [ ]:
df['Object ID'].dtype

In [ ]:
df['Orbit ID'].dtype

In [ ]:
df['Object ID'] = df['Object ID'].replace(['NaN', 'nan', 'NULL', 'null', ''], np.nan)

In [ ]:
df['Object ID'].isna().sum()


In [ ]:
df['Orbit ID'].isnull().sum()

In [ ]:
df['Orbit ID'] = pd.to_numeric(df['Orbit ID'], errors='coerce')
df['Orbit ID'] = df['Orbit ID'].astype('Int64')
print(df['Orbit ID'].dtype)

In [ ]:
print(df['Orbit ID'].isnull().sum())

In [ ]:
df[df['Orbit ID'].isnull()].index

In [ ]:
df.iloc[2874]

In [ ]:
df['Orbit ID'].sort_values()  # ascending


In [ ]:
df['Orbit ID']=df['Orbit ID'].fillna(0)
df['Orbit ID'].isnull().sum()

**2.Julian Data**

In [ ]:
df['Julian Date'] = pd.to_numeric(df['Julian Date'], errors='coerce')
df['Julian Date'] = df['Julian Date'].astype('float64')
print(df['Julian Date'].dtype)

In [ ]:
df['Julian Date'].unique()

In [ ]:
df['Julian Date'].isna().sum()

In [ ]:
df['Julian Date']=df['Julian Date'].round(2)
df.head(2)

**3.Close Approach Date**

In [ ]:
df['Close Approach Date'] = pd.to_datetime(df['Close Approach Date'], errors='coerce')

In [ ]:
df['Close Approach Date'].isna().sum()

In [ ]:
df['Year']=df['Close Approach Date'].dt.year
df['Month']=df['Close Approach Date'].dt.month
df['Day']=df['Close Approach Date'].dt.day
df['Time']=df['Close Approach Date'].dt.time
df=df.drop('Close Approach Date',axis=1)
df.head(2)

**4.Nominal Distance to Velocity infinity**

In [ ]:
columns_float=['Nominal Distance',
       'Min Distance', 'Max Distance', 'Relative Velocity',
       'Infinity Velocity']

In [ ]:
for i in columns_float:
  df[i]=pd.to_numeric(df[i],errors='coerce')
  df[i] = df[i].astype('float64')
  print(df[i].isna().sum())
  df[i]=df[i].round(3)

In [ ]:
df.head(4)

In [ ]:
df[df['Infinity Velocity'].isna()].index

In [ ]:
df['Infinity Velocity'].sort_values()

In [ ]:
df['Infinity Velocity']=df['Infinity Velocity'].fillna(0)
df.iloc[10661]

In [ ]:
df['Infinity Velocity'].isna().sum()

**6.Magnitude H**

In [ ]:
df['Magnitude H'].isna().sum()

In [ ]:
df[df['Magnitude H'].isna()].index

In [ ]:
df.iloc[2644]

In [ ]:
df['Magnitude H']=pd.to_numeric(df['Magnitude H'],errors='coerce')
df['Magnitude H'] = df['Magnitude H'].astype('float64')
df['Magnitude H']=df['Magnitude H'].fillna(df['Magnitude H'].median())

In [ ]:
df.iloc[2644]

**6.Time Uncertainity**

In [ ]:
df['Time Uncertainity'].unique()[:50]

In [ ]:
df['Time Uncertainity'].str.contains('<', na=False).sum()



In [ ]:
df['Time Uncertainity'].str.contains('_', na=False).sum()

In [ ]:
df['Time Uncertainity'].str.contains('>', na=False).sum()

In [ ]:
df['Time Uncertainity'].str.contains('~', na=False).sum()

In [ ]:
df['Time Uncertainity'] = df['Time Uncertainity'].str.replace('<', '', regex=True)
df['Time Uncertainity'] = df['Time Uncertainity'].str.strip()

In [ ]:
df['Time Uncertainity'].str.contains('<', na=False).sum()

In [ ]:
df['Time Uncertainity'] = df['Time Uncertainity'].str.replace('_', ':', regex=False)


In [ ]:
df['Time Uncertainity'].unique()[:50]

In [ ]:
def to_total_minutes(val):
    if pd.isna(val):
        return np.nan

    parts = val.strip().split(':')

    if len(parts) == 2:

        h, m = map(int, parts)
        return h * 60 + m

    elif len(parts) == 3:

        d, h, m = map(int, parts)
        return d * 1440 + h * 60 + m

    else:
        return np.nan

df['Time Uncertainty (minutes)'] = df['Time Uncertainity'].apply(to_total_minutes)


In [ ]:
df.head(4)

In [ ]:
df['Time Uncertainty (minutes)']=pd.to_numeric(df['Time Uncertainty (minutes)'],errors='coerce')
df['Time Uncertainty (minutes)'] = df['Time Uncertainty (minutes)'].astype('float64')


In [ ]:
df['Time Uncertainty (minutes)'].isna().sum()

**Time**

In [ ]:
df['Time'].dtype

In [ ]:


from datetime import datetime, time  # <- FIXED

def to_total_minutes(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val, pd.Timestamp):
        time_obj = val.time()
    elif isinstance(val, time):  # <- FIXED
        time_obj = val
    else:
        return np.nan

    h = time_obj.hour
    m = time_obj.minute
    s = time_obj.second

    return h * 60 + m + (s / 60.00)

# Example usage
df['Time(minutes)'] = df['Time'].apply(to_total_minutes)


In [ ]:
df.head(4)

In [ ]:
df['Time(minutes)']=pd.to_numeric(df['Time(minutes)'],errors='coerce')
df['Time(minutes)'] = df['Time(minutes)'].astype('float64')

In [ ]:
df['Time(minutes)'].isna().sum()

# Dropping unneccesary Columns and finally making the excel


In [ ]:
df=df.drop(['Time Uncertainity','Time'],axis=1)

In [ ]:
df.columns

In [ ]:
df=df.rename(columns={'Nominal Distance':'Nominal Distance(au)',
       'Min Distance':'Min Distance(au)', 'Max Distance':'Max Distance(au)', 'Relative Velocity':'Relative Velocity(km/s)','Infinity Velocity':'Infinity Velocity(km/s)','Time Uncertainty (minutes)':'Time Uncertainty (min)','Time(minutes)':'Time(min)'})

In [ ]:
df.head(4)

In [ ]:
df=df[['Object ID', 'Orbit ID', 'Julian Date', 'Year', 'Month', 'Day',
       'Nominal Distance(au)',
       'Min Distance(au)', 'Max Distance(au)', 'Relative Velocity(km/s)',
       'Infinity Velocity(km/s)', 'Magnitude H','Time Uncertainty (min)', 'Time(min)']]

In [ ]:
df.head(4)

In [ ]:
df.info()

In [ ]:
df = df.reset_index(drop=True)
df.index = df.index + 1
df.to_excel('NEO_NASA_datafinal.xlsx', index=True)


In [ ]:
df.head(4)

# Time Series Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
neo = pd.read_csv('NEO_NASA_datafinal.csv')
num_cols = ['Nominal Distance(au)', 'Relative Velocity(km/s)', 'Magnitude H']
neo[num_cols] = neo[num_cols].replace(0, pd.NA)

1.The rising blue curve reflects both genuine increases in encounters and, more importantly, the dramatic improvement in sky-survey coverage over the past two decades.

In [ ]:
# 1. Yearly close approaches trend
yearly = neo.groupby('Year').size().reset_index(name='count')
plt.figure(figsize=(9,4))
plt.plot(yearly['Year'], yearly['count'], color='steelblue')
plt.title('NEO close approaches per year (1900-2025)')
plt.xlabel('Year')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


2.The orange histogram shows a steep left-skew: encounters within 0.05 au (≈7.5 million km) are relatively rare, and the distribution tapers off sharply beyond 0.15 au.



In [ ]:
# 2. Histogram of nominal distance (au)
plt.figure(figsize=(6,4))
neo['Nominal Distance(au)'].dropna().clip(upper=0.2).hist(bins=40, color='darkorange')
plt.title('Distribution of close-approach distance (au)')
plt.xlabel('Nominal distance (au)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

3. The green histogram reveals a modal velocity around 10–15 km s⁻¹, but there’s a long tail up to ~70 km s⁻¹ where kinetic energy escalates enormously.

In [ ]:
# 3. Histogram of relative velocity
plt.figure(figsize=(6,4))
neo['Relative Velocity(km/s)'].dropna().hist(bins=40, color='seagreen')
plt.title('Distribution of relative velocity (km/s)')
plt.xlabel('Velocity (km/s)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

4.Putting it together: brightness vs. distance, sized by speed

Finally we relate physical size (brighter → probably larger) to encounter geometry. Each point’s size encodes velocity. Brighter, potentially larger NEOs (low H) are sprinkled across distances, yet some of the closest approaches are by faint, smaller bodies—highlighting why constant monitoring is essential.

In [ ]:
# 4. Magnitude vs nominal distance scatter (size ~ velocity)
plt.figure(figsize=(6,5))
subset = neo.sample(5000, random_state=1)  # sample for clarity
sizes = subset['Relative Velocity(km/s)'].fillna(0) * 2
sns.scatterplot(data=subset, x='Nominal Distance(au)', y='Magnitude H', size=sizes, hue=sizes, palette='viridis', legend=False, alpha=0.6)
plt.title('Brightness (H) vs. Distance, sized by velocity')
plt.xlabel('Nominal distance (au)')
plt.ylabel('Absolute magnitude H')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

insights = {}
insights['Total records'] = len(neo)
insights['Distinct NEO objects'] = neo['Object ID'].nunique()
insights['Year range'] = (neo['Year'].min(), neo['Year'].max())
insights['Average nominal distance (au)'] = float(neo['Nominal Distance(au)'].mean())
insights['Closest approach (au)'] = float(neo['Nominal Distance(au)'].min())
insights['Farthest approach (au)'] = float(neo['Nominal Distance(au)'].max())
insights['Average relative velocity (km/s)'] = float(neo['Relative Velocity(km/s)'].mean())
insights['Maximum relative velocity (km/s)'] = float(neo['Relative Velocity(km/s)'].max())
insights['Average magnitude H'] = float(neo['Magnitude H'].mean())

for i in insights:
  print(f"{i}: {insights[i]}")



In [ ]:

yearly = neo.groupby('Year').size().reset_index(name='close_approaches')

# Linear regression model
X = yearly['Year'].values.reshape(-1,1)
y = yearly['close_approaches'].values
model_lr = LinearRegression()
model_lr.fit(X, y)
R2=model_lr.score(X,y)
print(f"R2 Score: {R2}")

# Predict next 10 years beyond max year
last_year = yearly['Year'].max()
future_years = np.arange(last_year+1, last_year+11).reshape(-1,1)
forecast_counts = model_lr.predict(future_years).round().astype(int)
forecast_df = pd.DataFrame({'Year': future_years.flatten(), 'Predicted close approaches': forecast_counts})
print(forecast_df)

The linear-trend model explains about 44 % of the year-to-year variance in the number of recorded close approaches (R² ≈ 0.44). In other words, while the upward drift with time is real, more than half of the fluctuations are driven by additional factors—survey cadence changes, instrumentation upgrades, and random yearly variation—that a simple linear fit does not capture. If you need a higher-fidelity forecast we can experiment with richer time-series models (e.g. Prophet, GAMs, or Poisson regression with survey-effort covariates).

In [ ]:
# Plot historical and forecast
plt.figure(figsize=(10,4))
plt.plot(yearly['Year'], yearly['close_approaches'], marker='o', label='Historical')
plt.plot(forecast_df['Year'], forecast_df['Predicted close approaches'], marker='x', linestyle='--', label='Forecast')
plt.xlabel('Year')
plt.ylabel('NEO close approaches')
plt.title('Historical and Forecasted NEO close approaches per Year')
plt.legend()
plt.tight_layout()
plt.show()

What this shows??

The historical record reveals a steadily rising discovery/observation rate (partly due to better surveys).
Extrapolating that trend suggests roughly 1880–2000 recorded close approaches per year by 2035.
If you need a more sophisticated time-series model (e.g.\ Prophet with seasonality, incorporating discovery-bias corrections) or wish to forecast other variables such as minimum distance or velocity, let me know and we can refine the analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from statsmodels. tsa.api import ExponentialSmoothing, SimpleExpSmoothing, Holt
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
df1=pd.read_csv('monthly_approaches.csv')

In [ ]:
df1.head(4)

In [ ]:
df1['Date']=pd.to_datetime(df1['Date'])

In [ ]:
full_dates = pd.DataFrame({'Date': pd.date_range(start='1900-01-01', end='2025-12-30')})
df_full = full_dates.merge(df1, on='Date', how='left')
df_full['Approaches'] = df_full['Approaches'].fillna(0).astype(int)
df_full.set_index('Date', inplace=True)

In [ ]:
df_full.shape

In [ ]:
df_full.head(4)

In [ ]:
df_resampled = df_full['Approaches'].resample('M').sum()
plt.figure(figsize=(20,10))
sns.lineplot(data=df_resampled, label='Monthly Approaches', linewidth=2)

plt.title('Smoothed Curve of Approaches Over Time (1900–2025)', fontsize=16)
plt.xlabel('Month', fontsize=14)
plt.ylabel('Number of Approaches', fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df_resampled = df_full['Approaches'].resample('Y').sum()
plt.figure(figsize=(20,10))
sns.lineplot(data=df_resampled, label='Yearly Approaches', linewidth=2)

plt.title('Smoothed Curve of Approaches Over Time (1900–2025)', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Number of Approaches', fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df_resampled = df_full['Approaches'].resample('M').sum()
train=df_resampled[df_resampled.index.year<=2020]
test=df_resampled[df_resampled.index.year>2020]

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
train.plot(figsize=(20,10),fontsize=15)
test.plot(figsize=(20,10),fontsize=15)
plt.grid()
plt.xlabel('Month', fontsize=14)
plt.legend( ['Train','Test'])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Step 1: Prepare monthly data
df_resampled = df_full['Approaches'].resample('M').sum().reset_index()
df_resampled.columns = ['Date', 'Approaches']

# Step 2: Feature engineering
df_resampled['Year'] = df_resampled['Date'].dt.year
df_resampled['Month'] = df_resampled['Date'].dt.month
df_resampled['Month_sin'] = np.sin(2 * np.pi * df_resampled['Month'] / 12)
df_resampled['Month_cos'] = np.cos(2 * np.pi * df_resampled['Month'] / 12)
df_resampled['Day'] = df_resampled['Date'].dt.day
df_resampled['Quarter'] = df_resampled['Date'].dt.quarter
df_resampled['Days_since_1900'] = (df_resampled['Date'] - pd.Timestamp('1900-01-01')).dt.days
df_resampled['Lag_1'] = df_resampled['Approaches'].shift(1)
df_resampled['Rolling_mean_3'] = df_resampled['Approaches'].rolling(3).mean()

df_resampled.dropna(inplace=True)

# Step 3: Train-test split
train = df_resampled[df_resampled['Year'] <= 2020]
test = df_resampled[df_resampled['Year'] > 2020]

features = ['Year', 'Month_sin', 'Month_cos', 'Day', 'Quarter', 'Days_since_1900', 'Lag_1', 'Rolling_mean_3']
X_train = train[features]
y_train = train['Approaches']
X_test = test[features]
y_test = test['Approaches']

# Step 4: Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# Step 5: Evaluation
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("📊 Random Forest Evaluation with Advanced Features:")
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")



In [ ]:
# Future dates
future_dates = pd.date_range(start='2026-01-01', end='2030-12-01', freq='MS')
last_known = df_resampled.copy()
predicted_future = []

for date in future_dates:
    row = {
        'Date': date,
        'Year': date.year,
        'Month': date.month,
        'Month_sin': np.sin(2 * np.pi * date.month / 12),
        'Month_cos': np.cos(2 * np.pi * date.month / 12),
        'Day': date.day,
        'Quarter': date.quarter,
        'Days_since_1900': (date - pd.Timestamp('1900-01-01')).days,
    }

    # Lag + rolling from last_known
    lag_1 = last_known.iloc[-1]['Approaches']
    rolling_3 = last_known['Approaches'].iloc[-3:].mean()

    row['Lag_1'] = lag_1
    row['Rolling_mean_3'] = rolling_3

    X_future = pd.DataFrame([row])[features]
    prediction = rf_model.predict(X_future)[0]

    row['Approaches'] = prediction
    predicted_future.append(row)

    # Update last_known
    last_known = pd.concat([last_known, pd.DataFrame([row])], ignore_index=True)


In [ ]:
future_df = pd.DataFrame(predicted_future)

plt.figure(figsize=(20, 8))
plt.plot(test['Date'], y_test, label='Actual (2021–2025)', linewidth=2)
plt.plot(test['Date'], y_pred, label='Predicted (2021–2025)', linestyle='--', linewidth=2)
plt.plot(future_df['Date'], future_df['Approaches'], label='Forecast (2026–2030)', color='orange', linestyle='--', linewidth=2)
plt.title("Asteroid Approaches Forecast (Random Forest + Lag + Recursive)")
plt.xlabel("Date")
plt.ylabel("Approaches")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## Integrating RAG (Retrieval Augmented Generation)

To apply RAG, we need a knowledge base of text documents that the model can retrieve information from. For this demonstration, we'll create a small, mock knowledge base related to NEOs.

First, let's install the necessary library for creating text embeddings.

In [ ]:
pip install -U sentence-transformers google-generativeai

Next, let's set up the Gemini API. If you haven't already, ensure your `GOOGLE_API_KEY` is saved in Colab's secrets manager.

In [ ]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini API
gemini_model = genai.GenerativeModel('gemini-pro-latest')

Now, let's define our simple knowledge base (a list of documents/facts about NEOs) and create embeddings for them using `sentence-transformers`.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Our mock knowledge base about NEOs
documents = [
    "Near-Earth Objects (NEOs) are asteroids and comets that approach Earth's orbit.",
    "Most NEOs are asteroids, fragments of rock, metal, or combinations, that formed in the early solar system.",
    "Comets are icy bodies that release gas or dust, forming a visible atmosphere (coma) and sometimes a tail.",
    "The Sentry System at NASA's JPL continuously monitors the Earth for potential asteroid impacts over the next 100 years.",
    "The B612 Foundation is a private, non-profit foundation dedicated to planetary defense against asteroid impacts.",
    "The average size of a known NEO varies greatly, from a few meters to tens of kilometers.",
    "The Chelyabinsk meteor event in 2013 was caused by a superbolide that entered Earth's atmosphere over Russia."
]

# Load a pre-trained Sentence Transformer model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for our documents
document_embeddings = embedding_model.encode(documents, convert_to_tensor=True)

print(f"Created embeddings for {len(documents)} documents.")

Finally, let's create a RAG function that takes a query, retrieves the most relevant documents, and uses an LLM (Gemini) to generate an answer based on those retrieved documents.

In [ ]:
def rag_query(query, documents, document_embeddings, llm_model, top_k=2):
    # 1. Embed the query
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)

    # 2. Retrieve top-k most relevant documents
    similarities = cosine_similarity(query_embedding.unsqueeze(0), document_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1] # Get indices of top_k similarities
    retrieved_docs = [documents[i] for i in top_indices]

    # 3. Augment the prompt with retrieved documents
    context = "\n".join(retrieved_docs)
    prompt = f"""Based on the following information, answer the question:

Information:
{context}

Question: {query}
Answer:"""

    # 4. Generate response using the LLM
    response = llm_model.generate_content(prompt)
    return response.text

# Example Usage:
query_1 = "What are Near-Earth Objects?"
print(f"Query: {query_1}")
print(f"Answer: {rag_query(query_1, documents, document_embeddings, gemini_model)}\n")

query_2 = "Tell me about the Chelyabinsk event."
print(f"Query: {query_2}")
print(f"Answer: {rag_query(query_2, documents, document_embeddings, gemini_model)}\n")

query_3 = "What is the B612 Foundation?"
print(f"Query: {query_3}")
print(f"Answer: {rag_query(query_3, documents, document_embeddings, gemini_model)}\n")

In [ ]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Fetch key and configure
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize model
gemini_model = genai.GenerativeModel('gemini-pro-latest')

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Function to retrieve the most relevant documents based on user query
def retrieve_relevant_docs(query, top_k=2):
    # Generate embedding for the query
    query_embedding = embedding_model.encode([query])

    # Ensure document embeddings are on CPU / numpy array
    doc_embeddings_np = (
        document_embeddings.cpu().numpy()
        if hasattr(document_embeddings, 'cpu')
        else np.array(document_embeddings)
    )

    # Calculate cosine similarity between query and stored documents
    similarities = cosine_similarity(query_embedding, doc_embeddings_np)[0]

    # Rank indices by highest similarity
    top_indices = np.argsort(similarities)[::-1][:top_k]

    retrieved_texts = [documents[idx] for idx in top_indices]
    return retrieved_texts

# 2. Function to generate answer using retrieved context + Gemini
def answer_query_with_rag(query, top_k=2):
    # Retrieve relevant context
    retrieved_docs = retrieve_relevant_docs(query, top_k=top_k)
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])

    # Construct the RAG prompt
    prompt = f"""You are an astronomy assistant specializing in Near-Earth Objects (NEOs).
Answer the question accurately based ONLY on the provided context below.
If the context does not contain enough information to answer, state clearly that the answer is not available in the context.

Context:
{context}

Question:
{query}

Answer:"""

    # Generate response from Gemini
    response = gemini_model.generate_content(prompt)

    print(f"--- Retrieved Context ---")
    print(context)
    print(f"\n--- Model Response ---")
    print(response.text)

# --- Example Usage ---
query = "What is the Sentry System and what does it monitor?"
answer_query_with_rag(query)

In [ ]:
# List all available Gemini models that support 'generateContent'
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1. Load Cleaned Dataset
df = pd.read_excel('NEO_NASA_datafinal.xlsx')

# 2. Target Engineering (NASA PHA Criteria: Min Distance <= 0.05 AU & H <= 22.0)
df = df[df['Magnitude H'] > 0].copy()  # Filter out 0-imputed nulls for clean target
df['is_hazardous'] = (
    (df['Min Distance(au)'] <= 0.05) & (df['Magnitude H'] <= 22.0)
).astype(int)

# 3. Feature Engineering: Estimate Diameter (km)
# Formula: D = (1329 / sqrt(albedo)) * 10^(-0.2 * H), using typical asteroid albedo 0.14
df['Est_Diameter(km)'] = (1329 / np.sqrt(0.14)) * (10 ** (-0.2 * df['Magnitude H']))

# Select training features
features = [
    'Nominal Distance(au)',
    'Min Distance(au)',
    'Max Distance(au)',
    'Relative Velocity(km/s)',
    'Infinity Velocity(km/s)',
    'Magnitude H',
    'Est_Diameter(km)',
    'Time Uncertainty (min)',
]

X = df[features]
y = df['is_hazardous']

# 4. Train-Test Split (Stratified due to class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Handle Class Imbalance & Train XGBoost
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric='logloss',
)

xgb_model.fit(X_train, y_train)

# 6. Evaluation
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

# 7. SHAP Explainability
print("\nComputing SHAP values...")
explainer = shap.TreeExplainer(xgb_model)
shap_sample = X_test.sample(min(2000, len(X_test)), random_state=42)
shap_values = explainer(shap_sample)

# Save SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, shap_sample, show=False)
plt.title("SHAP Feature Importance for PHA Classification", fontsize=14)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=300)
plt.close()

# 8. Save Artifacts for Streamlit Dashboard
joblib.dump(xgb_model, 'pha_model.pkl')
joblib.dump(features, 'model_features.pkl')
print("Model and artifacts saved successfully as 'pha_model.pkl' and 'model_features.pkl'")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import joblib

# 1. Prepare Data
model_df = df.copy()
model_df = model_df[model_df['Magnitude H'] > 0].copy()

# 2. Define Real NASA Ground Truth
model_df['is_hazardous'] = (
    (model_df['Min Distance(au)'] <= 0.05) & (model_df['Magnitude H'] <= 22.0)
).astype(int)

# 3. Select Non-Leaking Predictive Features
features = [
    'Nominal Distance(au)',
    'Max Distance(au)',
    'Relative Velocity(km/s)',
    'Infinity Velocity(km/s)',
    'Time Uncertainty (min)',
    'Orbit ID',
    'Julian Date'
]

X = model_df[features]
y = model_df['is_hazardous']

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Handle Imbalance & Fit XGBoost
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# 6. Evaluate
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=== Clean (Non-Leaked) Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

# 7. SHAP Analysis
explainer = shap.TreeExplainer(xgb_model)
shap_sample = X_test.sample(min(1500, len(X_test)), random_state=42)
shap_values = explainer(shap_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, shap_sample, show=False)
plt.title("SHAP Feature Attribution (Leak-Free PHA Model)", fontsize=12)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=300)
plt.close()

# 8. Save Artifacts
joblib.dump(xgb_model, 'pha_model.pkl')
joblib.dump(features, 'model_features.pkl')
print("\nNew leak-free model artifacts saved successfully.")

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import streamlit as st

st.set_page_config(
    page_title="NASA NEO Planetary Hazard Intelligence",
    page_icon="☄️",
    layout="wide",
)

# 1. Load Data & Model
@st.cache_data
def load_data():
    df = pd.read_excel('NEO_NASA_datafinal.xlsx')
    if 'is_hazardous' not in df.columns:
        df['is_hazardous'] = (
            (df['Min Distance(au)'] <= 0.05) & (df['Magnitude H'] <= 22.0)
        ).astype(int)
    if 'Est_Diameter(km)' not in df.columns:
        df['Est_Diameter(km)'] = (1329 / np.sqrt(0.14)) * (10 ** (-0.2 * df['Magnitude H']))
    return df

@st.cache_resource
def load_model():
    model = joblib.load('pha_model.pkl')
    features = joblib.load('model_features.pkl')
    return model, features

df = load_data()
model, features = load_model()

# Header
st.title("☄️ NASA Near-Earth Objects (NEO) Hazard Intelligence")
st.markdown("Automated trajectory surveillance, risk profiling, and machine learning classification for 1900–2025 close approaches.")

# 2. Top-Level Metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Close Approaches", f"{len(df):,}")
col2.metric("Unique Asteroids Tracked", f"{df['Object ID'].nunique():,}")
col3.metric("Potentially Hazardous", f"{df['is_hazardous'].sum():,} ({df['is_hazardous'].mean()*100:.1f}%)")
col4.metric("Avg Approach Velocity", f"{df['Relative Velocity(km/s)'].mean():.2f} km/s")

st.divider()

# 3. Visual Analytics Section
tab1, tab2, tab3 = st.tabs(["📊 Exploratory Analytics", "🔮 Real-Time Risk Classifier", "🔍 Model Explainability (SHAP)"])

with tab1:
    col_left, col_right = st.columns(2)

    with col_left:
        st.subheader("Approach Frequency by Year")
        yearly_counts = df.groupby('Year').size().reset_index(name='Approaches')
        fig_year = px.line(yearly_counts, x='Year', y='Approaches', title="Approaches Over Time (1900–2025)", color_discrete_sequence=['#1f77b4'])
        st.plotly_chart(fig_year, use_container_width=True)

    with col_right:
        st.subheader("Velocity vs Proximity Risk Distribution")
        sample_df = df.sample(min(5000, len(df)), random_state=42)
        fig_scatter = px.scatter(
            sample_df,
            x='Min Distance(au)',
            y='Relative Velocity(km/s)',
            color=sample_df['is_hazardous'].map({1: 'Hazardous (PHA)', 0: 'Safe'}),
            color_discrete_map={'Hazardous (PHA)': '#d62728', 'Safe': '#2ca02c'},
            title="Proximity (AU) vs Relative Velocity (km/s)",
            opacity=0.6
        )
        st.plotly_chart(fig_scatter, use_container_width=True)

with tab2:
    st.subheader("Interactive Asteroid Hazard Inference")
    st.write("Input asteroid trajectory telemetry to assess real-time collision hazard probability.")

    c1, c2, c3 = st.columns(3)
    with c1:
        nom_dist = st.number_input("Nominal Distance (au)", value=0.045, min_value=0.0, step=0.005)
        min_dist = st.number_input("Min Distance (au)", value=0.038, min_value=0.0, step=0.005)
        max_dist = st.number_input("Max Distance (au)", value=0.052, min_value=0.0, step=0.005)
    with c2:
        rel_vel = st.number_input("Relative Velocity (km/s)", value=28.5, min_value=0.0, step=1.0)
        inf_vel = st.number_input("Infinity Velocity (km/s)", value=28.3, min_value=0.0, step=1.0)
        mag_h = st.number_input("Absolute Magnitude H", value=19.2, min_value=5.0, max_value=35.0, step=0.1)
    with c3:
        time_unc = st.number_input("Time Uncertainty (min)", value=15.0, min_value=0.0, step=1.0)
        est_diam = (1329 / np.sqrt(0.14)) * (10 ** (-0.2 * mag_h))
        st.metric("Computed Est. Diameter", f"{est_diam:.3f} km")

    if st.button("Evaluate Planetary Risk"):
        input_data = pd.DataFrame([[nom_dist, min_dist, max_dist, rel_vel, inf_vel, mag_h, est_diam, time_unc]], columns=features)
        pred = model.predict(input_data)[0]
        prob = model.predict_proba(input_data)[0][1]

        if pred == 1:
            st.error(f"⚠️ **HAZARD ALERT:** High Risk (PHA)! Confidence: {prob*100:.1f}%")
        else:
            st.success(f"✅ **SAFE:** Non-Hazardous Object. Safety Confidence: {(1-prob)*100:.1f}%")

with tab3:
    st.subheader("Global Feature Importance (SHAP)")
    st.image("shap_summary.png", caption="TreeSHAP Value Summary across features", use_container_width=True)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import joblib

st.set_page_config(page_title="NASA NEO Planetary Hazard Intelligence", page_icon="☄️", layout="wide")

@st.cache_data
def load_data():
    return pd.read_parquet('neo_cleaned_data.parquet')

@st.cache_resource
def load_model():
    model = joblib.load('pha_model.pkl')
    features = joblib.load('model_features.pkl')
    return model, features

df = load_data()
model, features = load_model()

st.title("☄️ NASA Near-Earth Objects (NEO) Hazard Intelligence")
st.markdown("Automated risk classification and trajectory surveillance dashboard.")

# Top KPIs
c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Close Approaches", f"{len(df):,}")
c2.metric("Unique Asteroids", f"{df['Object ID'].nunique():,}")
c3.metric("Hazardous (PHA)", f"{df['is_hazardous'].sum():,} ({df['is_hazardous'].mean()*100:.1f}%)")
c4.metric("Avg Velocity", f"{df['Relative Velocity(km/s)'].mean():.2f} km/s")

st.divider()

tab1, tab2, tab3 = st.tabs(["📊 Analytics", "🔮 Real-Time Classifier", "🔍 Model Explainability (SHAP)"])

with tab1:
    col_left, col_right = st.columns(2)
    with col_left:
        st.subheader("Approach Frequency by Year")
        yearly_counts = df.groupby('Year').size().reset_index(name='Approaches')
        fig_year = px.line(yearly_counts, x='Year', y='Approaches', title="Approaches (1900–2025)", color_discrete_sequence=['#1f77b4'])
        st.plotly_chart(fig_year, use_container_width=True)
    with col_right:
        st.subheader("Proximity vs Velocity Distribution")
        sample_df = df.sample(min(3000, len(df)), random_state=42)
        fig_scatter = px.scatter(
            sample_df,
            x='Min Distance(au)',
            y='Relative Velocity(km/s)',
            color=sample_df['is_hazardous'].map({1: 'Hazardous (PHA)', 0: 'Safe'}),
            color_discrete_map={'Hazardous (PHA)': '#d62728', 'Safe': '#2ca02c'},
            title="Min Distance (au) vs Velocity (km/s)",
            opacity=0.6
        )
        st.plotly_chart(fig_scatter, use_container_width=True)

with tab2:
    st.subheader("Asteroid Hazard Prediction (Leak-Free Model)")
    st.write("Input non-leaked trajectory metrics to evaluate collision hazard level:")

    col1, col2, col3 = st.columns(3)
    with col1:
        nom_dist = st.number_input("Nominal Distance (au)", value=0.045, step=0.005)
        max_dist = st.number_input("Max Distance (au)", value=0.052, step=0.005)
    with col2:
        rel_vel = st.number_input("Relative Velocity (km/s)", value=28.5, step=1.0)
        inf_vel = st.number_input("Infinity Velocity (km/s)", value=28.3, step=1.0)
    with col3:
        time_unc = st.number_input("Time Uncertainty (min)", value=15.0, step=1.0)
        orbit_id = st.number_input("Orbit ID", value=16, step=1)
        julian_date = st.number_input("Julian Date", value=2450000.0, step=100.0)

    if st.button("Evaluate Hazard"):
        input_data = pd.DataFrame([[nom_dist, max_dist, rel_vel, inf_vel, time_unc, orbit_id, julian_date]], columns=features)
        pred = model.predict(input_data)[0]
        prob = model.predict_proba(input_data)[0][1]

        if pred == 1:
            st.error(f"⚠️ **HAZARD ALERT:** High Risk (PHA)! Model Probability: {prob*100:.1f}%")
        else:
            st.success(f"✅ **SAFE:** Non-Hazardous Object. Safety Probability: {(1-prob)*100:.1f}%")

with tab3:
    st.subheader("Feature Importance (SHAP TreeExplainer)")
    st.image("shap_summary.png", caption="Global SHAP Attribution Summary")

In [ ]:
import urllib.request

external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print(f"👉 Tunnel Password / Endpoint IP: {external_ip}")
print("=" * 60)

# Run Streamlit in background and launch localtunnel with -y flag to avoid prompt freeze
!streamlit run app.py & npx -y localtunnel --port 8501

In [ ]:
import subprocess
import time
import urllib.request

# 1. Get Colab Public IP (Password for LocalTunnel)
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print(f"👉 Tunnel Password / Endpoint IP: {external_ip}")
print("=" * 60)

# 2. Launch Streamlit as a persistent background process
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)

# 3. Launch LocalTunnel
!npx -y localtunnel --port 8501

In [ ]:
# 1. Kill old streamlit and localtunnel processes
!fuser -k 8501/tcp
!pkill -f streamlit
!pkill -f localtunnel

# 2. Verify all 4 required files exist
import os
required_files = ['app.py', 'pha_model.pkl', 'model_features.pkl', 'neo_cleaned_data.parquet', 'shap_summary.png']
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f"❌ Missing files: {missing}. Please make sure you ran the data save & training cells.")
else:
    print("✅ All required files are present on disk!")

In [ ]:
import subprocess
import time
import urllib.request

# Print IP password
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print(f"👉 Tunnel Password / Endpoint IP: {ip}")
print("=" * 60)

# Launch Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)

# Launch tunnel
!npx -y localtunnel --port 8501

In [ ]:
# Download and install cloudflared in Colab
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import subprocess
import time
import re

# 1. Kill old processes
!fuser -k 8501/tcp
!pkill -f streamlit
!pkill -f cloudflared

# 2. Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)

# 3. Start Cloudflare Tunnel and read public URL
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Parse and print URL
for line in tunnel_process.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        print("\n" + "=" * 60)
        print(f"🚀 Live Interactive Dashboard URL:\n👉 {match.group(0)}")
        print("=" * 60 + "\n")
        break

In [ ]:
# 1. Install pyarrow / fastparquet to support parquet format
!pip install pyarrow fastparquet -q

# 2. Add the target & diameter columns to your existing in-memory dataframe and save as parquet
df_save = df.copy()
df_save = df_save[df_save['Magnitude H'] > 0].copy()
df_save['is_hazardous'] = (
    (df_save['Min Distance(au)'] <= 0.05) & (df_save['Magnitude H'] <= 22.0)
).astype(int)
df_save['Est_Diameter(km)'] = (1329 / np.sqrt(0.14)) * (10 ** (-0.2 * df_save['Magnitude H']))

# Save as parquet
df_save.to_parquet('neo_cleaned_data.parquet', index=False)
print("✅ Saved 'neo_cleaned_data.parquet' successfully!")

In [ ]:
!pip install streamlit